In [1]:
# ============================================
# EDA FOR RecipeDB_general.csv
# Publication-quality plots for paper
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

# ============================================
# LOAD DATA
# ============================================

file_path = r"./dataset/RecipeDB_general.csv"

df = pd.read_csv(file_path)

print(df.shape)
print(df.head())

# ============================================
# CREATE OUTPUT FOLDER
# ============================================

output_dir = Path("eda_figures")
output_dir.mkdir(exist_ok=True)

# ============================================
# PAPER-QUALITY FIGURE SETTINGS
# ============================================

mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 12
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

sns.set_style("white")  # removes grid background

# ============================================
# CLEAN COLUMN NAMES
# ============================================

df.columns = df.columns.str.strip()

# ============================================
# BASIC INFO
# ============================================

print("\n========== INFO ==========\n")
print(df.info())

print("\n========== MISSING VALUES ==========\n")
print(df.isnull().sum())

print("\n========== DUPLICATES ==========\n")
print("Duplicate rows:", df.duplicated().sum())

# ============================================
# NUMERICAL COLUMNS
# ============================================

numerical_cols = [
    'Calories',
    'cook_time',
    'prep_time',
    'servings',
    'total_time',
    'Carbohydrate, by difference (g)',
    'Energy (kcal)',
    'Protein (g)',
    'Total lipid (fat) (g)'
]

# Keep only existing columns
numerical_cols = [c for c in numerical_cols if c in df.columns]

# Convert to numeric safely
for col in numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ============================================
# DESCRIPTIVE STATISTICS
# ============================================

print("\n========== DESCRIPTIVE STATS ==========\n")
print(df[numerical_cols].describe())

# ============================================
# HISTOGRAMS
# ============================================

for col in numerical_cols:

    fig, ax = plt.subplots(figsize=(6,4))

    ax.hist(df[col].dropna(), bins=30)

    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / f"{col}_histogram.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# BOXPLOTS
# ============================================

for col in numerical_cols:

    fig, ax = plt.subplots(figsize=(5,3))

    ax.boxplot(df[col].dropna(), vert=False)

    ax.set_xlabel(col)

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / f"{col}_boxplot.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# CORRELATION HEATMAP
# ============================================

corr = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(8,6))

im = ax.imshow(corr, aspect='auto')

ax.set_xticks(np.arange(len(numerical_cols)))
ax.set_yticks(np.arange(len(numerical_cols)))

ax.set_xticklabels(numerical_cols, rotation=45, ha='right')
ax.set_yticklabels(numerical_cols)

cbar = plt.colorbar(im)

ax.grid(False)

plt.tight_layout()

plt.savefig(
    output_dir / "correlation_heatmap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.close()

# ============================================
# CATEGORICAL ANALYSIS
# ============================================

categorical_cols = [
    'Region',
    'Sub_region',
    'Continent',
    'Source'
]

categorical_cols = [c for c in categorical_cols if c in df.columns]

for col in categorical_cols:

    top_categories = df[col].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(8,5))

    ax.bar(top_categories.index.astype(str), top_categories.values)

    ax.set_xlabel(col)
    ax.set_ylabel("Count")

    plt.xticks(rotation=45, ha='right')

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / f"{col}_top_categories.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# DIETARY FLAGS ANALYSIS
# ============================================

diet_cols = [
    'vegan',
    'pescetarian',
    'ovo_vegetarian',
    'lacto_vegetarian',
    'ovo_lacto_vegetarian'
]

diet_cols = [c for c in diet_cols if c in df.columns]

for col in diet_cols:

    counts = df[col].value_counts(dropna=False)

    fig, ax = plt.subplots(figsize=(5,4))

    ax.bar(counts.index.astype(str), counts.values)

    ax.set_xlabel(col)
    ax.set_ylabel("Count")

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / f"{col}_distribution.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# TOP REGIONS BY RECIPE COUNT
# ============================================

if 'Region' in df.columns:

    top_regions = df['Region'].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(9,5))

    ax.bar(top_regions.index.astype(str), top_regions.values)

    ax.set_xlabel("Region")
    ax.set_ylabel("Recipe Count")

    plt.xticks(rotation=45, ha='right')

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / "top_regions.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# COOK TIME VS CALORIES
# ============================================

if 'cook_time' in df.columns and 'Calories' in df.columns:

    fig, ax = plt.subplots(figsize=(6,5))

    ax.scatter(
        df['cook_time'],
        df['Calories'],
        alpha=0.6
    )

    ax.set_xlabel("Cook Time")
    ax.set_ylabel("Calories")

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / "cooktime_vs_calories.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# SERVINGS VS CALORIES
# ============================================

if 'servings' in df.columns and 'Calories' in df.columns:

    fig, ax = plt.subplots(figsize=(6,5))

    ax.scatter(
        df['servings'],
        df['Calories'],
        alpha=0.6
    )

    ax.set_xlabel("Servings")
    ax.set_ylabel("Calories")

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / "servings_vs_calories.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# RECIPE TITLE LENGTH DISTRIBUTION
# ============================================

if 'Recipe_title' in df.columns:

    df['title_length'] = df['Recipe_title'].astype(str).apply(len)

    fig, ax = plt.subplots(figsize=(6,4))

    ax.hist(df['title_length'], bins=30)

    ax.set_xlabel("Recipe Title Length")
    ax.set_ylabel("Frequency")

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / "recipe_title_length.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# MISSING VALUES VISUALIZATION
# ============================================

missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) > 0:

    fig, ax = plt.subplots(figsize=(10,5))

    ax.bar(missing.index.astype(str), missing.values)

    ax.set_xlabel("Columns")
    ax.set_ylabel("Missing Values")

    plt.xticks(rotation=90)

    ax.grid(False)

    plt.tight_layout()

    plt.savefig(
        output_dir / "missing_values.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================
# SAVE CLEAN SUMMARY
# ============================================

summary = df.describe(include='all')

summary.to_csv(output_dir / "eda_summary.csv")

print("\n===================================")
print("EDA COMPLETED SUCCESSFULLY")
print("All figures saved in:", output_dir)
print("===================================")

C:\Users\hvish\AppData\Local\Temp\ipykernel_45136\4359585.py:19: DtypeWarning: Columns (2,3,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


(118083, 24)
   Recipe_id  Calories cook_time prep_time servings  \
0       2610     196.0        30        15        4   
1       2611      80.0        35        10        4   
2       2612     339.0       120        15        4   
3       2613     409.0        15        15        3   
4       2614      45.0         5        20       24   

                        Recipe_title total_time  \
0               Egyptian Lentil Soup         45   
1  Egyptian Green Beans with Carrots         45   
2                     Egyptian Bamia        135   
3        Magpie's Easy Falafel Cakes         60   
4                             Dukkah         25   

                                                 url          Region  \
0  http://allrecipes.com/recipe/222661/egyptian-l...  Middle Eastern   
1  http://allrecipes.com/recipe/233456/egyptian-g...  Middle Eastern   
2  http://allrecipes.com/recipe/227986/egyptian-b...  Middle Eastern   
3  http://allrecipes.com/recipe/143113/magpies-ea...  Middle 

In [3]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# ==========================================
# LOAD DATA
# ==========================================

file_path = r'./protocol_manual_classification_updated.csv'

df = pd.read_csv(file_path)

# ==========================================
# CLEAN COLUMN NAMES
# ==========================================

df.columns = df.columns.str.strip()

# ==========================================
# BASIC CHECK
# ==========================================

print(df.head())
print(df.columns)
print(df.shape)

# ==========================================
# REMOVE MISSING VALUES
# ==========================================

df = df[['Word', 'REMARK']].dropna()

# ==========================================
# CREATE WORD LENGTH FEATURE
# ==========================================

df['word_length'] = df['Word'].astype(str).apply(len)

# ==========================================
# CREATE OUTPUT FOLDER
# ==========================================

output_dir = Path('manual_cluster_figures')
output_dir.mkdir(exist_ok=True)

# ==========================================
# PAPER-QUALITY SETTINGS
# ==========================================

mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 12
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'

# ==========================================
# BOXPLOT FOR ALL CLUSTERS
# ==========================================

clusters = sorted(df['REMARK'].unique())

cluster_data = [
    df[df['REMARK'] == cluster]['word_length']
    for cluster in clusters
]

fig, ax = plt.subplots(figsize=(10, 6))

ax.boxplot(cluster_data, labels=clusters)

ax.set_xlabel('Clusters')
ax.set_ylabel('Word Length')

ax.grid(False)

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(
    output_dir / 'manual_clusters_boxplot.png',
    dpi=300,
    bbox_inches='tight'
)

plt.close()

# ==========================================
# CLUSTER SIZE DISTRIBUTION
# ==========================================

cluster_counts = df['REMARK'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

ax.set_xlabel('Clusters')
ax.set_ylabel('Number of Words')

ax.grid(False)

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(
    output_dir / 'cluster_distribution.png',
    dpi=300,
    bbox_inches='tight'
)

plt.close()

# ==========================================
# OPTIONAL: TOP WORDS PER CLUSTER
# ==========================================

for cluster in clusters:

    words = df[df['REMARK'] == cluster]['Word'].head(10)

    print(f'\nCluster: {cluster}')
    print(words.tolist())

# ==========================================
# DONE
# ==========================================

print('\n===================================')
print('Plots saved successfully.')
print('Location:', output_dir)
print('===================================')



       Word         REMARK
0  splutter  cooking sound
1   deflate   shape change
2    stream         action
3     crimp   shape change
4   floured         action
Index(['Word', 'REMARK'], dtype='object')
(270, 2)


C:\Users\hvish\AppData\Local\Temp\ipykernel_45136\522595658.py:70: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(cluster_data, labels=clusters)



Cluster: action
['stream', 'floured', 'select', 'tilt', 'store', 'insert', 'handle', 'shake', 'squeeze', 'seal']

Cluster: add sugar
['sweeten']

Cluster: add water
['absorbed', 'baste', 'moisten', 'soak', 'submerge', 'spray', 'immerse', 'dissolve', 'splash', 'moist']

Cluster: circular/flip action
['invert', 'twist', 'swirl', 'whirl', 'turn']

Cluster: cleaning
['devein', 'sterilize', 'scrub', 'scrape', 'rinse', 'wipe', 'wash', 'clean']

Cluster: coating
['caramelize', 'grease', 'dredge', 'sear', 'caramelized', 'glaze', 'slather', 'spread', 'coat', 'butter']

Cluster: cooking sound
['splutter', 'foam', 'bubbling', 'sizzle', 'bubble']

Cluster: cooling
['freezing', 'ice', 'chill', 'cool']

Cluster: cutting
['slit', 'pierce', 'prick', 'flake', 'crumble', 'snap', 'grind', 'blitz', 'slash', 'carve']

Cluster: decoration
['presentation', 'decorate', 'style', 'garnish', 'scatter', 'sort', 'prepare', 'lard', 'pack', 'serve']

Cluster: filtering
['sieve', 'strain']

Cluster: flavoring
['stee